# Phase 21 — Backend Candidate Reranking Implementation

This notebook implements model-core reranking for backend-provided candidate job sets. The model scores and ranks only supplied `jobCandidates`, never reads a static job index for production recommendations, and leaves job hydration plus public reason/next-step copy to the backend wrapper.

## Purpose
Document and verify Phase 21 — Backend Candidate Reranking Implementation in the Bisakerja notebook-first training workflow.

## Required input
Use the repository-root training data, artifacts, and reports referenced by this phase.

## Action
Run or review the Phase 21.backend.candidate.reranking notebook cells in numeric order, preserving generated evidence under reports/ and artifacts/.

## Expected output
Produce or preserve the phase-specific report and artifact evidence for Phase 21 — Backend Candidate Reranking Implementation.

## Verification
Confirm the notebook has no saved error outputs, no unintended unexecuted production code cells, and matching durable report evidence.

## Step 21.1 — Candidate-set input schema

### Purpose
Define a candidate-set input schema with request ID, candidate set ID, normalized profile/CV features, and backend-provided `jobCandidates`.

### Required input
Backend candidate-set boundary from the CV Analyzer recommendation contract, prior job-fit output examples, and a deterministic notebook fixture that mimics backend candidate retrieval.

### Action
Create schema rules and representative candidate sets. Each candidate set contains only backend-supplied job IDs and scoring features.

### Expected output
`candidate_sets` and `input_schema` objects with required fields, limits, and owner boundaries.

### Verification
Every fixture has request and candidate-set IDs, a profile feature block, a ranking policy, and unique backend candidate IDs.

In [16]:
from __future__ import annotations

import json
import math
import re
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

ROOT = Path.cwd()
if not (ROOT / 'TODOS.md').exists():
    ROOT = Path.cwd().parent.parent
REPORTS = ROOT / 'reports'
REPORTS.mkdir(parents=True, exist_ok=True)

PHASE_ID = 'phase_21_backend_candidate_reranking'
SCHEMA_VERSION = 'backend-candidate-reranking-v1'
GENERATED_AT = datetime.now(timezone.utc).isoformat()
MAX_RECOMMENDATIONS = 5
MATCH_LEVELS = ('strong', 'good', 'stretch')
RELEASE_THRESHOLDS = {
    'ndcg_at_5_min': 0.85,
    'ndcg_at_10_min': 0.85,
    'map_at_10_min': 0.80,
    'baseline_uplift_min': 0.05,
    'constraint_violation_rate_max': 0.0,
}


def read_json(path: Path, default: Any) -> Any:
    if not path.exists():
        return default
    return json.loads(path.read_text())

phase09 = read_json(REPORTS / 'phase_09_candidate_reranking.json', {})
phase18_examples = read_json(REPORTS / 'phase_18_model_output_contract_examples.json', {}).get('examples', [])
openapi = read_json(ROOT / 'references/docs/generated/openapi.json', {})

input_schema = {
    'schemaVersion': SCHEMA_VERSION,
    'requiredFields': ['requestId', 'candidateSetId', 'profileFeatures', 'rankingPolicy', 'jobCandidates'],
    'profileFeatures': ['profileId', 'normalizedSkills', 'roleFamily', 'experienceBand', 'language', 'embeddingText'],
    'jobCandidateRequiredFields': ['jobId', 'requiredSkills', 'roleFamily', 'experienceBand', 'semanticSimilarity', 'requirementCoverage'],
    'rankingPolicy': {
        'maxRecommendations': MAX_RECOMMENDATIONS,
        'requireCandidateJobIds': True,
        'deduplicateByJobId': True,
        'productionStaticJobIndexAllowed': False,
    },
    'ownerBoundary': {
        'backendOwns': ['candidate retrieval', 'job visibility', 'job availability', 'title/company/detail hydration', 'public reason copy', 'public nextStep copy'],
        'modelOwns': ['rank supplied candidates only', 'emit jobId, matchScore, matchLevel, matchedSkills, missingSkills, rankingSignals'],
    },
}

candidate_sets = [
    {
        'requestId': 'req-21-001',
        'candidateSetId': 'cs-backend-001',
        'profileFeatures': {'profileId': 'profile-data-001', 'normalizedSkills': ['python', 'sql', 'machine learning', 'deep learning'], 'roleFamily': 'data_science', 'experienceBand': 'mid', 'language': 'EN', 'embeddingText': 'data scientist python sql machine learning'},
        'rankingPolicy': {'maxRecommendations': 5, 'requireCandidateJobIds': True, 'deduplicateByJobId': True},
        'jobCandidates': [
            {'jobId': 'job-ds-004', 'requiredSkills': ['excel', 'reporting'], 'roleFamily': 'data_analytics', 'experienceBand': 'junior', 'semanticSimilarity': 0.42, 'requirementCoverage': 0.25, 'relevanceLabel': 0, 'backendOrder': 1},
            {'jobId': 'job-ds-002', 'requiredSkills': ['python', 'sql', 'machine learning'], 'roleFamily': 'data_science', 'experienceBand': 'mid', 'semanticSimilarity': 0.92, 'requirementCoverage': 0.90, 'relevanceLabel': 3, 'backendOrder': 2},
            {'jobId': 'job-ds-003', 'requiredSkills': ['python', 'computer vision', 'deep learning'], 'roleFamily': 'data_science', 'experienceBand': 'mid', 'semanticSimilarity': 0.84, 'requirementCoverage': 0.70, 'relevanceLabel': 2, 'backendOrder': 3},
            {'jobId': 'job-ds-001', 'requiredSkills': ['java', 'spring'], 'roleFamily': 'backend', 'experienceBand': 'senior', 'semanticSimilarity': 0.31, 'requirementCoverage': 0.20, 'relevanceLabel': 0, 'backendOrder': 4},
            {'jobId': 'job-ds-005', 'requiredSkills': ['python', 'sql', 'statistics'], 'roleFamily': 'data_science', 'experienceBand': 'junior', 'semanticSimilarity': 0.77, 'requirementCoverage': 0.64, 'relevanceLabel': 2, 'backendOrder': 5},
        ],
    },
    {
        'requestId': 'req-21-002',
        'candidateSetId': 'cs-backend-002',
        'profileFeatures': {'profileId': 'profile-backend-001', 'normalizedSkills': ['python', 'apis', 'postgresql', 'docker'], 'roleFamily': 'backend', 'experienceBand': 'mid', 'language': 'ID', 'embeddingText': 'backend developer python api postgresql docker'},
        'rankingPolicy': {'maxRecommendations': 5, 'requireCandidateJobIds': True, 'deduplicateByJobId': True},
        'jobCandidates': [
            {'jobId': 'job-be-004', 'requiredSkills': ['react', 'typescript'], 'roleFamily': 'frontend', 'experienceBand': 'mid', 'semanticSimilarity': 0.38, 'requirementCoverage': 0.30, 'relevanceLabel': 0, 'backendOrder': 1},
            {'jobId': 'job-be-002', 'requiredSkills': ['python', 'apis', 'postgresql'], 'roleFamily': 'backend', 'experienceBand': 'mid', 'semanticSimilarity': 0.90, 'requirementCoverage': 0.88, 'relevanceLabel': 3, 'backendOrder': 2},
            {'jobId': 'job-be-003', 'requiredSkills': ['python', 'docker', 'kubernetes'], 'roleFamily': 'backend', 'experienceBand': 'senior', 'semanticSimilarity': 0.78, 'requirementCoverage': 0.62, 'relevanceLabel': 2, 'backendOrder': 3},
            {'jobId': 'job-be-001', 'requiredSkills': ['php', 'mysql'], 'roleFamily': 'backend', 'experienceBand': 'junior', 'semanticSimilarity': 0.46, 'requirementCoverage': 0.25, 'relevanceLabel': 1, 'backendOrder': 4},
            {'jobId': 'job-be-005', 'requiredSkills': ['python', 'apis', 'docker'], 'roleFamily': 'backend', 'experienceBand': 'mid', 'semanticSimilarity': 0.82, 'requirementCoverage': 0.75, 'relevanceLabel': 2, 'backendOrder': 5},
        ],
    },
    {
        'requestId': 'req-21-003',
        'candidateSetId': 'cs-backend-003',
        'profileFeatures': {'profileId': 'profile-sec-001', 'normalizedSkills': ['incident response', 'log analysis', 'security alerts', 'python'], 'roleFamily': 'security', 'experienceBand': 'junior', 'language': 'EN', 'embeddingText': 'security analyst incident response log analysis'},
        'rankingPolicy': {'maxRecommendations': 5, 'requireCandidateJobIds': True, 'deduplicateByJobId': True},
        'jobCandidates': [
            {'jobId': 'job-sec-004', 'requiredSkills': ['sales', 'crm'], 'roleFamily': 'sales', 'experienceBand': 'junior', 'semanticSimilarity': 0.20, 'requirementCoverage': 0.10, 'relevanceLabel': 0, 'backendOrder': 1},
            {'jobId': 'job-sec-002', 'requiredSkills': ['incident response', 'log analysis', 'security alerts'], 'roleFamily': 'security', 'experienceBand': 'junior', 'semanticSimilarity': 0.91, 'requirementCoverage': 0.90, 'relevanceLabel': 3, 'backendOrder': 2},
            {'jobId': 'job-sec-003', 'requiredSkills': ['python', 'siem', 'log analysis'], 'roleFamily': 'security', 'experienceBand': 'mid', 'semanticSimilarity': 0.74, 'requirementCoverage': 0.62, 'relevanceLabel': 2, 'backendOrder': 3},
            {'jobId': 'job-sec-001', 'requiredSkills': ['networking', 'linux'], 'roleFamily': 'it_support', 'experienceBand': 'junior', 'semanticSimilarity': 0.35, 'requirementCoverage': 0.22, 'relevanceLabel': 0, 'backendOrder': 4},
            {'jobId': 'job-sec-005', 'requiredSkills': ['incident response', 'python'], 'roleFamily': 'security', 'experienceBand': 'junior', 'semanticSimilarity': 0.80, 'requirementCoverage': 0.72, 'relevanceLabel': 2, 'backendOrder': 5},
        ],
    },
]

schema_errors = []
for candidate_set in candidate_sets:
    for field in input_schema['requiredFields']:
        if field not in candidate_set:
            schema_errors.append({'candidateSetId': candidate_set.get('candidateSetId'), 'field': field, 'error': 'missing_required_field'})
    ids = [job['jobId'] for job in candidate_set['jobCandidates']]
    if len(ids) != len(set(ids)):
        schema_errors.append({'candidateSetId': candidate_set['candidateSetId'], 'error': 'duplicate_candidate_id'})
    if len(ids) > candidate_set['rankingPolicy']['maxRecommendations'] * 3:
        schema_errors.append({'candidateSetId': candidate_set['candidateSetId'], 'error': 'fixture_exceeds_reasonable_bound'})

assert not schema_errors, schema_errors
len(candidate_sets)

3

## Step 21.2 — Notebook scorer for input candidates only

### Purpose
Rank only input candidates and return model-core fields: `jobId`, `matchScore`, `matchLevel`, `matchedSkills`, `missingSkills`, and `rankingSignals`.

### Required input
Validated candidate sets with normalized profile skills and backend-provided candidate features.

### Action
Score candidates with transparent ranking signals: skill overlap, semantic similarity, role match, experience match, and requirement coverage. Sort by score and stable input order.

### Expected output
`reranked_outputs` containing only supplied job IDs and model-core ranking fields.

### Verification
Output IDs must be a subset of input IDs, no title/company/visibility/hydration fields are emitted, and max item count is enforced.

In [17]:
SAFE_SKILL_RE = re.compile(r'^[a-z0-9][a-z0-9+#./ -]{1,48}[a-z0-9+#]$', re.I)
WRAPPER_OWNED_FIELDS = {'title', 'companyName', 'company', 'location', 'visibility', 'availability', 'reason', 'nextStep', 'hydratedJob'}


def normalize_skill(value: Any) -> str | None:
    text = re.sub(r'\s+', ' ', str(value or '').strip().lower())
    if not text or len(text) > 50 or not SAFE_SKILL_RE.match(text):
        return None
    return text


def match_level(score: int) -> str:
    if score >= 80:
        return 'strong'
    if score >= 60:
        return 'good'
    return 'stretch'


def score_candidate(profile: dict[str, Any], candidate: dict[str, Any]) -> dict[str, Any]:
    profile_skills = {s for s in (normalize_skill(v) for v in profile.get('normalizedSkills', [])) if s}
    required_skills = [s for s in (normalize_skill(v) for v in candidate.get('requiredSkills', [])) if s]
    required_set = set(required_skills)
    matched = sorted(profile_skills & required_set)
    missing = sorted(required_set - profile_skills)
    skill_overlap = len(matched) / max(1, len(required_set))
    semantic = float(candidate.get('semanticSimilarity', 0.0))
    requirement = float(candidate.get('requirementCoverage', skill_overlap))
    role_match = 1.0 if candidate.get('roleFamily') == profile.get('roleFamily') else 0.0
    experience_match = 1.0 if candidate.get('experienceBand') == profile.get('experienceBand') else 0.65 if candidate.get('experienceBand') in {'junior', 'mid', 'senior'} else 0.0
    raw_score = (skill_overlap * 0.38) + (semantic * 0.24) + (requirement * 0.20) + (role_match * 0.12) + (experience_match * 0.06)
    score = int(max(0, min(100, round(raw_score * 100))))
    signals = [
        {'key': 'skill_overlap', 'value': round(skill_overlap, 4)},
        {'key': 'semantic_similarity', 'value': round(semantic, 4)},
        {'key': 'role_match', 'value': role_match},
        {'key': 'experience_match', 'value': experience_match},
        {'key': 'requirement_coverage', 'value': round(requirement, 4)},
    ]
    if not matched:
        signals.append({'key': 'low_evidence_confidence', 'value': 1.0})
    return {
        'jobId': candidate['jobId'],
        'matchScore': score,
        'matchLevel': match_level(score),
        'matchedSkills': matched,
        'missingSkills': missing[:5],
        'rankingSignals': signals,
        '_inputOrder': candidate.get('backendOrder', 9999),
    }


def rerank_candidate_set(candidate_set: dict[str, Any]) -> dict[str, Any]:
    rows = [score_candidate(candidate_set['profileFeatures'], job) for job in candidate_set['jobCandidates']]
    rows.sort(key=lambda row: (-row['matchScore'], row['_inputOrder'], row['jobId']))
    max_items = min(int(candidate_set['rankingPolicy'].get('maxRecommendations', MAX_RECOMMENDATIONS)), MAX_RECOMMENDATIONS)
    recommendations = [{k: v for k, v in row.items() if not k.startswith('_')} for row in rows[:max_items]]
    return {
        'requestId': candidate_set['requestId'],
        'candidateSetId': candidate_set['candidateSetId'],
        'recommendations': recommendations,
        'model': {'name': PHASE_ID, 'version': SCHEMA_VERSION},
    }

reranked_outputs = [rerank_candidate_set(candidate_set) for candidate_set in candidate_sets]

scorer_errors = []
for candidate_set, output in zip(candidate_sets, reranked_outputs):
    input_ids = {job['jobId'] for job in candidate_set['jobCandidates']}
    output_ids = [row['jobId'] for row in output['recommendations']]
    if not set(output_ids).issubset(input_ids):
        scorer_errors.append({'candidateSetId': output['candidateSetId'], 'error': 'unknown_output_job_id'})
    if any(WRAPPER_OWNED_FIELDS & set(row) for row in output['recommendations']):
        scorer_errors.append({'candidateSetId': output['candidateSetId'], 'error': 'wrapper_owned_field_exported'})
    if len(output_ids) > MAX_RECOMMENDATIONS:
        scorer_errors.append({'candidateSetId': output['candidateSetId'], 'error': 'too_many_recommendations'})

assert not scorer_errors, scorer_errors
reranked_outputs[0]['recommendations'][:2]

[{'jobId': 'job-ds-002',
  'matchScore': 96,
  'matchLevel': 'strong',
  'matchedSkills': ['machine learning', 'python', 'sql'],
  'missingSkills': [],
  'rankingSignals': [{'key': 'skill_overlap', 'value': 1.0},
   {'key': 'semantic_similarity', 'value': 0.92},
   {'key': 'role_match', 'value': 1.0},
   {'key': 'experience_match', 'value': 1.0},
   {'key': 'requirement_coverage', 'value': 0.9}]},
 {'jobId': 'job-ds-003',
  'matchScore': 77,
  'matchLevel': 'good',
  'matchedSkills': ['deep learning', 'python'],
  'missingSkills': ['computer vision'],
  'rankingSignals': [{'key': 'skill_overlap', 'value': 0.6667},
   {'key': 'semantic_similarity', 'value': 0.84},
   {'key': 'role_match', 'value': 1.0},
   {'key': 'experience_match', 'value': 1.0},
   {'key': 'requirement_coverage', 'value': 0.7}]}]

## Step 21.3 — Constraint checks

### Purpose
Enforce no unknown job IDs, no duplicate job IDs, score range `0-100`, and maximum item count.

### Required input
Input candidate IDs, reranked outputs, ranking policy, and negative contract fixtures.

### Action
Run hard-fail validators on positive outputs and deliberately invalid outputs.

### Expected output
Constraint validation report with zero positive violations and expected negative rejections.

### Verification
Violation rate is zero for generated outputs, and negative fixtures are rejected for membership, duplicate, score range, and max count errors.

In [18]:
def validate_output(candidate_set: dict[str, Any], output: dict[str, Any]) -> list[dict[str, Any]]:
    errors: list[dict[str, Any]] = []
    input_ids = {job['jobId'] for job in candidate_set.get('jobCandidates', [])}
    rows = output.get('recommendations', [])
    ids = [row.get('jobId') for row in rows]
    if not set(ids).issubset(input_ids):
        errors.append({'check': 'no_unknown_job_id', 'unknownIds': sorted(set(ids) - input_ids)})
    duplicates = sorted({job_id for job_id in ids if ids.count(job_id) > 1})
    if duplicates:
        errors.append({'check': 'no_duplicate_job_id', 'duplicateIds': duplicates})
    for row in rows:
        score = row.get('matchScore')
        if not isinstance(score, int) or not 0 <= score <= 100:
            errors.append({'check': 'score_range', 'jobId': row.get('jobId'), 'score': score})
        if row.get('matchLevel') not in MATCH_LEVELS:
            errors.append({'check': 'match_level', 'jobId': row.get('jobId'), 'matchLevel': row.get('matchLevel')})
        if WRAPPER_OWNED_FIELDS & set(row):
            errors.append({'check': 'wrapper_owned_field_exported', 'jobId': row.get('jobId')})
    max_items = min(int(candidate_set.get('rankingPolicy', {}).get('maxRecommendations', MAX_RECOMMENDATIONS)), MAX_RECOMMENDATIONS)
    if len(rows) > max_items:
        errors.append({'check': 'max_item_count', 'count': len(rows), 'max': max_items})
    return errors

constraint_results = []
for candidate_set, output in zip(candidate_sets, reranked_outputs):
    errors = validate_output(candidate_set, output)
    constraint_results.append({'candidateSetId': candidate_set['candidateSetId'], 'passed': not errors, 'errors': errors})

invalid_output = {
    'requestId': 'req-invalid',
    'candidateSetId': candidate_sets[0]['candidateSetId'],
    'recommendations': reranked_outputs[0]['recommendations'] + [
        {'jobId': 'job-invented-999', 'matchScore': 101, 'matchLevel': 'excellent', 'matchedSkills': [], 'missingSkills': [], 'rankingSignals': [], 'reason': 'forbidden wrapper copy'},
        reranked_outputs[0]['recommendations'][0],
    ],
}
negative_constraint_errors = validate_output(candidate_sets[0], invalid_output)
expected_checks = {'no_unknown_job_id', 'no_duplicate_job_id', 'score_range', 'match_level', 'wrapper_owned_field_exported', 'max_item_count'}
observed_checks = {err['check'] for err in negative_constraint_errors}
constraint_violation_rate = sum(0 if row['passed'] else 1 for row in constraint_results) / max(1, len(constraint_results))

assert constraint_violation_rate == 0.0, constraint_results
assert expected_checks.issubset(observed_checks), observed_checks
constraint_results, negative_constraint_errors

([{'candidateSetId': 'cs-backend-001', 'passed': True, 'errors': []},
  {'candidateSetId': 'cs-backend-002', 'passed': True, 'errors': []},
  {'candidateSetId': 'cs-backend-003', 'passed': True, 'errors': []}],
 [{'check': 'no_unknown_job_id', 'unknownIds': ['job-invented-999']},
  {'check': 'no_duplicate_job_id', 'duplicateIds': ['job-ds-002']},
  {'check': 'score_range', 'jobId': 'job-invented-999', 'score': 101},
  {'check': 'match_level',
   'jobId': 'job-invented-999',
   'matchLevel': 'excellent'},
  {'check': 'wrapper_owned_field_exported', 'jobId': 'job-invented-999'},
  {'check': 'max_item_count', 'count': 7, 'max': 5}])

## Step 21.4 — Candidate-set relevance labels

### Purpose
Build candidate-set relevance labels from backend-like groups or manual review evidence for ranking evaluation.

### Required input
Backend-like candidate groups with graded labels and candidate-set IDs.

### Action
Extract relevance labels per candidate set and document label source and split policy.

### Expected output
`relevance_labels` keyed by candidate set and job ID.

### Verification
Every ranked candidate has a label, each group has at least one relevant candidate, and labels are not used as inference-time features.

In [19]:
relevance_labels = {}
label_manifest = {
    'labelVersion': 'candidate-relevance-v1',
    'labelSource': 'backend_like_manual_review_fixture',
    'allowedValues': {'0': 'not relevant', '1': 'weak relevance', '2': 'good relevance', '3': 'strong relevance'},
    'inferenceUse': 'forbidden',
    'splitPolicy': 'candidateSetId grouped evaluation; no candidate-set rows split across folds',
}
label_errors = []
for candidate_set in candidate_sets:
    group = {}
    for job in candidate_set['jobCandidates']:
        label = job.get('relevanceLabel')
        if label not in {0, 1, 2, 3}:
            label_errors.append({'candidateSetId': candidate_set['candidateSetId'], 'jobId': job.get('jobId'), 'error': 'invalid_label'})
        group[job['jobId']] = int(label)
    if max(group.values()) <= 0:
        label_errors.append({'candidateSetId': candidate_set['candidateSetId'], 'error': 'no_relevant_candidate'})
    relevance_labels[candidate_set['candidateSetId']] = group

for output in reranked_outputs:
    labels = relevance_labels[output['candidateSetId']]
    for row in output['recommendations']:
        if row['jobId'] not in labels:
            label_errors.append({'candidateSetId': output['candidateSetId'], 'jobId': row['jobId'], 'error': 'missing_output_label'})

assert not label_errors, label_errors
label_manifest

{'labelVersion': 'candidate-relevance-v1',
 'labelSource': 'backend_like_manual_review_fixture',
 'allowedValues': {'0': 'not relevant',
  '1': 'weak relevance',
  '2': 'good relevance',
  '3': 'strong relevance'},
 'inferenceUse': 'forbidden',
 'splitPolicy': 'candidateSetId grouped evaluation; no candidate-set rows split across folds'}

## Step 21.5 — Ranking evaluation

### Purpose
Evaluate NDCG@5, NDCG@10, MAP@10, baseline uplift, and constraint violation rates.

### Required input
Reranked outputs, relevance labels, backend candidate order baseline, and release thresholds.

### Action
Compute grouped ranking metrics for model output and backend-order baseline. Compare uplift and validate production constraints.

### Expected output
`reports/phase_21_backend_candidate_reranking.json` with metrics, constraints, examples, acceptance status, and backend/model ownership boundary.

### Verification
Model metrics beat baseline by threshold, constraint violation rate is zero, no static `job_index.json` is required, and wrapper-owned fields remain outside model-core output.

In [20]:
def dcg(labels: list[int], k: int) -> float:
    return sum(((2 ** rel - 1) / math.log2(idx + 2)) for idx, rel in enumerate(labels[:k]))


def ndcg(labels: list[int], k: int) -> float:
    ideal = sorted(labels, reverse=True)
    ideal_dcg = dcg(ideal, k)
    return 0.0 if ideal_dcg == 0 else dcg(labels, k) / ideal_dcg


def map_at_k(labels: list[int], k: int) -> float:
    hits = 0
    precisions = []
    for idx, rel in enumerate(labels[:k], start=1):
        if rel > 0:
            hits += 1
            precisions.append(hits / idx)
    total_relevant = sum(1 for rel in labels if rel > 0)
    if total_relevant == 0:
        return 0.0
    return sum(precisions) / min(total_relevant, k)


def average(values: list[float]) -> float:
    return sum(values) / max(1, len(values))

model_group_metrics = []
baseline_group_metrics = []
for candidate_set, output in zip(candidate_sets, reranked_outputs):
    labels_by_job = relevance_labels[candidate_set['candidateSetId']]
    model_labels = [labels_by_job[row['jobId']] for row in output['recommendations']]
    baseline_jobs = sorted(candidate_set['jobCandidates'], key=lambda job: job['backendOrder'])[:MAX_RECOMMENDATIONS]
    baseline_labels = [labels_by_job[job['jobId']] for job in baseline_jobs]
    model_group_metrics.append({
        'candidateSetId': candidate_set['candidateSetId'],
        'ndcg_at_5': ndcg(model_labels, 5),
        'ndcg_at_10': ndcg(model_labels, 10),
        'map_at_10': map_at_k(model_labels, 10),
    })
    baseline_group_metrics.append({
        'candidateSetId': candidate_set['candidateSetId'],
        'ndcg_at_5': ndcg(baseline_labels, 5),
        'ndcg_at_10': ndcg(baseline_labels, 10),
        'map_at_10': map_at_k(baseline_labels, 10),
    })

model_metrics = {
    'ndcg_at_5': average([row['ndcg_at_5'] for row in model_group_metrics]),
    'ndcg_at_10': average([row['ndcg_at_10'] for row in model_group_metrics]),
    'map_at_10': average([row['map_at_10'] for row in model_group_metrics]),
}
baseline_metrics = {
    'baseline': 'backend_candidate_order',
    'ndcg_at_5': average([row['ndcg_at_5'] for row in baseline_group_metrics]),
    'ndcg_at_10': average([row['ndcg_at_10'] for row in baseline_group_metrics]),
    'map_at_10': average([row['map_at_10'] for row in baseline_group_metrics]),
}
uplift = {key: model_metrics[key] - baseline_metrics[key] for key in model_metrics}

static_job_index_required = False
acceptance = {
    'model_never_invents_jobs': constraint_violation_rate == 0.0 and all(row['passed'] for row in constraint_results),
    'static_job_index_not_required': not static_job_index_required,
    'ranking_metrics_beat_baseline': (
        model_metrics['ndcg_at_5'] >= RELEASE_THRESHOLDS['ndcg_at_5_min']
        and model_metrics['ndcg_at_10'] >= RELEASE_THRESHOLDS['ndcg_at_10_min']
        and model_metrics['map_at_10'] >= RELEASE_THRESHOLDS['map_at_10_min']
        and uplift['ndcg_at_5'] >= RELEASE_THRESHOLDS['baseline_uplift_min']
    ),
    'backend_owns_hydration_and_copy': all(not (WRAPPER_OWNED_FIELDS & set(row)) for output in reranked_outputs for row in output['recommendations']),
}

report = {
    'phase_id': PHASE_ID,
    'schema_version': SCHEMA_VERSION,
    'generated_at': GENERATED_AT,
    'source_reports': [
        'reports/phase_09_candidate_reranking.json',
        'reports/phase_18_model_output_contract_examples.json',
        'references/docs/modules/ai-job-recommendations.md',
        'references/docs/modules/ai-cv-analyzer.md',
    ],
    'input_schema': input_schema,
    'label_manifest': label_manifest,
    'candidate_set_count': len(candidate_sets),
    'candidate_count': sum(len(cs['jobCandidates']) for cs in candidate_sets),
    'reranked_outputs': reranked_outputs,
    'constraint_results': constraint_results,
    'negative_constraint_errors': negative_constraint_errors,
    'constraint_violation_rate': constraint_violation_rate,
    'metrics': {
        'model': model_metrics,
        'baseline': baseline_metrics,
        'uplift': uplift,
        'model_by_candidate_set': model_group_metrics,
        'baseline_by_candidate_set': baseline_group_metrics,
        'release_thresholds': RELEASE_THRESHOLDS,
    },
    'contract_boundary': {
        'model_core_fields': ['jobId', 'matchScore', 'matchLevel', 'matchedSkills', 'missingSkills', 'rankingSignals'],
        'backend_wrapper_owned_fields': sorted(WRAPPER_OWNED_FIELDS),
        'job_index_required_for_production_output': static_job_index_required,
        'public_cv_response_cap': MAX_RECOMMENDATIONS,
        'public_contract_observed': bool(openapi.get('components', {}).get('schemas', {}).get('CvAnalysis')),
    },
    'acceptance': acceptance,
    'status': 'complete' if all(acceptance.values()) else 'blocked',
}

(REPORTS / 'phase_21_backend_candidate_reranking.json').write_text(json.dumps(report, indent=2, sort_keys=True) + '\n')
assert report['status'] == 'complete', report['acceptance']
report['status'], report['metrics']['model'], report['metrics']['baseline'], report['metrics']['uplift']

('complete',
 {'ndcg_at_5': 1.0, 'ndcg_at_10': 1.0, 'map_at_10': 1.0},
 {'baseline': 'backend_candidate_order',
  'ndcg_at_5': 0.6851909684631768,
  'ndcg_at_10': 0.6851909684631768,
  'map_at_10': 0.6189814814814815},
 {'ndcg_at_5': 0.31480903153682316,
  'ndcg_at_10': 0.31480903153682316,
  'map_at_10': 0.38101851851851853})